## EU REED

#### 1) Import packages and set filepath

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tarfile
from pathlib import Path
from reed import utils
from collections import defaultdict
from functools import reduce

In [ ]:
config = utils.Config()
#test

In [ ]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"

#### 2) Import foreground database 

In [ ]:
all_records = []

for company_folder in monitoring_data_path.iterdir():
    if not company_folder.is_dir():
        continue

    parts = [p.strip() for p in company_folder.name.split(" - ", 1)]
    if len(parts) != 2:
        print("Advarsel – unexpected name of folder:", company_folder.name)
        continue

    company, machine = parts

    # Find .tar.gz
    tar_files = list(company_folder.glob("*.tar.gz"))
    # Hvis dine filer evt. ligger i undermapper:
    # tar_files = list(company_folder.rglob("*.tar.gz"))

    if not tar_files:
        print(f"Ingen .tar.gz-filer i {company_folder.name}")
        continue

    for tar_path in tar_files:
        stem = tar_path.name.replace(".tar.gz", "")  # fx OF10001_G_BQC_S8CF2G_7N4ZJ8
        try:
            event_id, measurement_type = stem.rsplit("_", 1)
        except ValueError:
            event_id, measurement_type = stem, None

        with tarfile.open(tar_path, "r:gz") as tf:  # ← vigtigt: r:gz
            csv_members = [m for m in tf.getmembers()
                           if m.name.lower().endswith(".csv")]

            if not csv_members:
                print(f"Ingen CSV fundet i {tar_path.name}")
                continue

            csv_member = csv_members[0]
            with tf.extractfile(csv_member) as f:
                df = pd.read_csv(f)

        all_records.append({
            "company": company,
            "maskinenavn": machine,
            "event_id": event_id,
            "measurement_type": measurement_type,
            "tar_path": tar_path,
            "csv_name": csv_member.name,
            "data": df,
        })

print("\nTotal amount of excelsheets:", len(all_records))

# === group company and event ===
events = defaultdict(list)
for rec in all_records:
    events[(rec["company"], rec["event_id"])].append(rec)

print("Events:", len(events))


In [ ]:
def merge_records(records):
    """
    Creating one dataFrame for each event.

    """
    dfs = []
    cols = ["timestamp", "Tool_Number", "Power_Active", "Energy_Total"]

    merged = None
    
    for i, r in enumerate(records):
        df = r["data"].copy()
        df = df[[c for c in cols if c in df.columns]].copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.floor('s')
        df = df.set_index("timestamp")

        if merged is None:
            merged = df
        else:
            merged = merged.join(df, how="outer")

    merged = merged.sort_index()
            
    return merged

In [ ]:
all_grouped = []  
event_summaries = []
for (company, event_id), records in events.items():
    merged_df = merge_records(records)

    merged_df["energy_consumption"] = merged_df["Power_Active"] / 3600

    merged_df = merged_df.reset_index()

    s = merged_df["Energy_Total"].dropna()

    if s.empty:
        meter_energy = np.nan
    else:
        meter_energy = s.iloc[-1] - s.iloc[0]

    event_summaries.append({
        "company": company,
        "event_id": event_id,
        "energy_meter_total": meter_energy,
    })

    def tool_hours(ts):
        ts = ts.sort_values()
        if ts.empty:
            return 0.0
        return (ts.iloc[-1] - ts.iloc[0]).total_seconds() / 3600.0

    # Group after tool_number
    grouped_df = (
        merged_df
        .groupby("Tool_Number", as_index=False)
        .agg(
            energy_consumption=("energy_consumption", "sum"),
            Power_Active=("Power_Active", "mean"),
            seconds=("timestamp", "count"),
            )
        )
    
    grouped_df["hours"] = grouped_df["seconds"] / 3600.0
    grouped_df["company"] = company
    grouped_df["event_id"] = event_id

    grouped_df = grouped_df.set_index(["company", "event_id", "Tool_Number"])

    all_grouped.append(grouped_df)

final_df = pd.concat(all_grouped)

In [ ]:
print(final_df)

In [ ]:
df_totals = (
    final_df
    .reset_index()  # sikrer at company, event_id og Tool_Number er kolonner
    .groupby(["company", "event_id"], as_index=False)
    .agg({
        "energy_consumption": "sum",   
        "Power_Active": "mean",
        "hours": "sum"        
    })
    .set_index(["company", "event_id"])  # hvis du vil have dem som index igen
)

df_meter = (
    pd.DataFrame(event_summaries)
    .set_index(["company", "event_id"])
)

df_totals = df_totals.join(df_meter)

df_totals["rel_diff_calc_vs_meter"] = (
    (df_totals["energy_consumption"] - df_totals["energy_meter_total"])
    / df_totals["energy_meter_total"]
)




In [ ]:
print(df_totals)